# 03-short. Colab de novo PPG 대안모델 + Optuna

이 노트북은 공개 repo를 내려받지 않고, Colab 런타임 안에서 직접 실행합니다.

흐름:
1. 필요한 Python 패키지만 설치
2. Figshare에서 LLaMAC zip 다운로드
3. PPG / rich-PPG feature 생성
4. Logistic Regression, LightGBM, ExtraTrees, HistGradientBoosting 비교
5. Logistic Regression / LightGBM 짧은 Optuna 탐색

기본값은 빠른 확인용입니다. 전체 실험은 `LIMIT_SUBJECTS=None`, `FOLDS=5`, `OPTUNA_TRIALS=20+`로 바꾸세요.


## 0. 독립 실행 환경 준비


In [ ]:
# Colab 내부에서 독립 실행되도록 필요한 패키지와 작업 폴더만 준비합니다.
# 공개 repo를 내려받지 않습니다. 필요한 코드는 이 노트북 셀 안에 들어 있습니다.
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal, Sequence

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Colab 런타임 안에 독립 작업 폴더를 만듭니다.
    ROOT = Path("/content/llamac_denovo")
    ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)
    print("Colab 패키지를 설치합니다. 처음 실행할 때 1~3분 정도 걸릴 수 있습니다.")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "polars>=1.40.1",
            "pyarrow>=14",
            "numpy>=2.0",
            "scipy>=1.14",
            "scikit-learn>=1.5",
            "lightgbm>=4.5",
            "optuna>=4.0",
            "matplotlib>=3.8",
        ],
        check=True,
    )
else:
    # 로컬 Jupyter에서는 현재 작업 폴더를 root로 사용합니다.
    ROOT = Path.cwd().resolve()

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

RAW_DIR = ROOT / "data" / "raw"
EXTRACTED_DIR = ROOT / "data" / "extracted"
PROCESSED_DIR = ROOT / "data" / "processed"
ARTIFACT_DIR = ROOT / "artifacts"
INDEX_PATH = PROCESSED_DIR / "dataset_index.csv"
SKIP_HEAVY = os.environ.get("LLAMAC_SKIP_HEAVY") == "1"  # 자동 검증용: 다운로드/학습 생략

for path in [RAW_DIR, EXTRACTED_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"ROOT = {ROOT}")


## 1. 실험 설정


In [ ]:
# 빠른 Colab 데모 기본값입니다.
# 전체 실험은 LIMIT_SUBJECTS=None, FOLDS=5, OPTUNA_TRIALS=20 이상으로 바꾸세요.
LIMIT_SUBJECTS = 12
FOLDS = 3
SEED = 42
DOWNLOAD_WORKERS = 4
FEATURE_WORKERS = 1
DEVICE = "auto"
OPTUNA_TRIALS = 5

REBUILD_FEATURES = False
RERUN_BASELINES = False
RERUN_OPTUNA = False

FEATURE_DIR = PROCESSED_DIR / "short_colab_denovo"
RESULT_DIR = ARTIFACT_DIR / "short_colab_denovo" / "ppg_alternatives"
OPTUNA_DIR = ARTIFACT_DIR / "short_colab_denovo" / "optuna"
FEATURE_PPG = FEATURE_DIR / "features_ppg_short.parquet"
FEATURE_PPG_RICH = FEATURE_DIR / "features_ppg_rich_short.parquet"
for path in [FEATURE_DIR, RESULT_DIR, OPTUNA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"LIMIT_SUBJECTS={LIMIT_SUBJECTS}, FOLDS={FOLDS}, DEVICE={DEVICE}, OPTUNA_TRIALS={OPTUNA_TRIALS}")


## 2. LLaMAC 데이터셋 다운로드와 압축해제


In [ ]:
# Figshare API로 LLaMAC zip 파일을 직접 다운로드하고 안전하게 압축해제합니다.
# LIMIT_SUBJECTS가 12이면 처음 12개 participant zip만 받습니다.
ARTICLE_ID = "28748696"
VERSION = "6"
API_URL = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/versions/{VERSION}"


@dataclass(frozen=True)
class FigshareFile:
    name: str
    size: int
    download_url: str
    md5: str | None = None


def natural_key(text: str | Path) -> list[object]:
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(text))]


def safe_name(name: str) -> str:
    # zip 내부/파일명 경로 탈출을 막기 위해 파일명만 남깁니다.
    out = name.replace("\\", "/").split("/")[-1]
    if not out or out in {".", ".."}:
        raise ValueError(f"unsafe filename: {name!r}")
    return out


def fetch_figshare_files() -> list[FigshareFile]:
    req = urllib.request.Request(API_URL, headers={"User-Agent": "llamac-colab-denovo/1.0"})
    with urllib.request.urlopen(req, timeout=60) as response:
        metadata = json.load(response)
    files = []
    for item in metadata["files"]:
        files.append(
            FigshareFile(
                name=str(item["name"]),
                size=int(item["size"]),
                download_url=str(item["download_url"]),
                md5=item.get("computed_md5") or item.get("supplied_md5"),
            )
        )
    (RAW_DIR / "llamac_figshare_manifest.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
    return files


def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def download_one(file: FigshareFile) -> str:
    target = RAW_DIR / safe_name(file.name)
    if target.exists() and target.stat().st_size == file.size:
        return f"skip {file.name}"
    tmp = target.with_suffix(target.suffix + ".part")
    req = urllib.request.Request(file.download_url, headers={"User-Agent": "llamac-colab-denovo/1.0"})
    with urllib.request.urlopen(req, timeout=180) as response, tmp.open("wb") as out:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    if tmp.stat().st_size != file.size:
        raise RuntimeError(f"size mismatch for {file.name}")
    if file.md5 and md5sum(tmp) != file.md5:
        raise RuntimeError(f"md5 mismatch for {file.name}")
    tmp.replace(target)
    return f"download {file.name}"


def safe_extract_zip(zip_path: Path, target_dir: Path) -> None:
    marker = target_dir / ".extracted_ok"
    if marker.exists():
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            member = Path(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"unsafe zip member: {info.filename}")
        zf.extractall(target_dir)
    marker.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


def build_dataset_index() -> pl.DataFrame:
    rows = []
    for subject_dir in sorted([p for p in EXTRACTED_DIR.iterdir() if p.is_dir()], key=natural_key):
        for path in sorted(subject_dir.rglob("*"), key=natural_key):
            if not path.is_file() or path.name == ".extracted_ok":
                continue
            rows.append(
                {
                    "participant_id": subject_dir.name,
                    "file_name": path.name,
                    "relative_path": str(path.relative_to(EXTRACTED_DIR)),
                    "suffix": path.suffix.lower(),
                    "size_bytes": path.stat().st_size,
                }
            )
    df = pl.DataFrame(rows)
    df.write_csv(INDEX_PATH)
    return df


if INDEX_PATH.exists():
    print(f"이미 준비된 index 사용: {INDEX_PATH}")
    index = pl.read_csv(INDEX_PATH)
elif SKIP_HEAVY:
    print("LLAMAC_SKIP_HEAVY=1 이므로 다운로드를 생략합니다.")
    index = pl.DataFrame()
else:
    fig_files = fetch_figshare_files()
    zips = sorted([f for f in fig_files if f.name.lower().endswith(".zip")], key=lambda f: natural_key(f.name))
    selected = zips[:LIMIT_SUBJECTS] if LIMIT_SUBJECTS is not None else zips
    print(f"다운로드 zip 수: {len(selected)}")

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_one, f) for f in selected]
        for i, future in enumerate(as_completed(futures), start=1):
            print(f"[{i}/{len(selected)}] {future.result()}")

    for f in selected:
        zip_path = RAW_DIR / safe_name(f.name)
        safe_extract_zip(zip_path, EXTRACTED_DIR / zip_path.stem)
    index = build_dataset_index()

if index.height:
    print(index.select(pl.len().alias("files"), pl.col("participant_id").n_unique().alias("participants")))


## 3. 짧은 EDA: 데이터 구조, raw waveform, label 관계

여기서는 모델링 전에 자료구조만 빠르게 확인합니다.

확인 항목:
- participant별 어떤 CSV 파일들이 있는지
- `band_*.csv` 한 파일이 어떤 dataframe 구조인지
- GSR/PPG/SKT raw waveform이 대략 정상적으로 보이는지
- `NoVideo`에서 만든 intended label과 참가자가 답한 reported label이 어떻게 다른지
- Valence/Arousal/Dominance/Liking 같은 rating 값과 label의 상관관계

참고: Valence/Arousal 등은 예측 목표로 만들 수 있습니다. 회귀나 ordinal classification 대상이 됩니다. 다만 이 값들은 안정적인 성격 특성이라기보다 각 trial 직후의 주관적 정서 rating에 가깝고, PPG-only로는 노이즈가 클 수 있어 이 짧은 노트북에서는 주 목표로 쓰지 않고 EDA에서만 확인합니다.


In [ ]:

# 간단 EDA 함수들입니다. 복잡한 feature engineering 전, 원자료의 모양을 눈으로 확인합니다.
EDA_LIMIT_SUBJECTS = LIMIT_SUBJECTS


def eda_file_family(name: str) -> str:
    """파일명을 사람이 보기 쉬운 family 이름으로 묶습니다."""
    if name == "answer.csv":
        return "answer"
    if name.startswith("band_"):
        return "band"
    if name.startswith("eeg_"):
        return "eeg"
    if name.startswith("respiration_"):
        return "respiration"
    return "other"


def intended_from_novideo(no_video: int | float | None) -> int | None:
    """NoVideo 자극 번호를 의도된 감정 label 1~5로 변환합니다."""
    if no_video is None:
        return None
    no_video = int(no_video)
    if 1 <= no_video <= 10:
        return 1
    if 11 <= no_video <= 20:
        return 2
    if 21 <= no_video <= 30:
        return 3
    if 31 <= no_video <= 40:
        return 4
    if 41 <= no_video <= 50:
        return 5
    return None


EDA_EMOTION_NAMES = {1: "neutral", 2: "fun", 3: "sadness", 4: "anger", 5: "fear"}


def add_eda_labels(frame: pl.DataFrame) -> pl.DataFrame:
    """answer table에 intended/reported label과 사람이 읽는 label 이름을 추가합니다."""
    return frame.with_columns(
        pl.col("NoVideo").map_elements(intended_from_novideo, return_dtype=pl.Int64).alias("IntendedType"),
        pl.col("EmotType").cast(pl.Int64, strict=False).alias("ReportedType"),
    ).with_columns(
        pl.col("IntendedType").replace_strict(EDA_EMOTION_NAMES, default="unknown").alias("IntendedLabel"),
        pl.col("ReportedType").replace_strict(EDA_EMOTION_NAMES, default="unknown").alias("ReportedLabel"),
    )


def load_answers_for_eda(index: pl.DataFrame) -> pl.DataFrame:
    """각 participant의 answer.csv를 모아서 trial-level label table을 만듭니다."""
    if index.height == 0:
        return pl.DataFrame()
    answer_index = index.filter(pl.col("file_name") == "answer.csv").sort("participant_id")
    if EDA_LIMIT_SUBJECTS is not None:
        answer_index = answer_index.head(EDA_LIMIT_SUBJECTS)
    tables = []
    for row in answer_index.iter_rows(named=True):
        answer_path = EXTRACTED_DIR / row["relative_path"]
        tables.append(pl.read_csv(answer_path).with_columns(pl.lit(int(row["participant_id"])).alias("SubjectID")))
    if not tables:
        return pl.DataFrame()
    return add_eda_labels(pl.concat(tables, how="diagonal_relaxed"))


if index.height == 0:
    print("index가 비어 있어 EDA를 건너뜁니다. 다운로드 셀을 먼저 실행하세요.")
    answers_eda = pl.DataFrame()
else:
    # 1) 파일 구조 요약: participant마다 answer/band/eeg/respiration 파일이 몇 개 있는지 봅니다.
    index_eda = index.with_columns(
        pl.col("file_name").map_elements(eda_file_family, return_dtype=pl.Utf8).alias("family")
    )
    family_summary = (
        index_eda.group_by("family")
        .agg(
            pl.len().alias("file_count"),
            pl.col("participant_id").n_unique().alias("participants"),
            (pl.col("size_bytes").sum() / 1024**2).round(2).alias("size_mib"),
        )
        .sort("family")
    )
    display(family_summary)

    # 2) band sensor CSV 한 개의 dataframe 형태를 직접 봅니다.
    band_row = index_eda.filter(pl.col("family") == "band").sort(["participant_id", "file_name"]).row(0, named=True)
    band_path = EXTRACTED_DIR / band_row["relative_path"]
    band_df = pl.read_csv(band_path)
    print(f"sample band file: participant={band_row['participant_id']}, file={band_row['file_name']}")
    print(f"band_df shape = {band_df.shape}")
    print(f"band_df columns = {band_df.columns}")
    display(band_df.head(8))

    # 3) raw waveform sanity plot: GSR, PPG, SKT가 시간축에 따라 정상적인 모양인지 봅니다.
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=False)
    for ax, value_col, time_col in zip(axes, ["GSR", "PPG", "SKT"], ["GSR_Time", "PPG_Time", "SKT_Time"], strict=True):
        t = band_df[time_col].to_numpy()
        y = band_df[value_col].to_numpy()
        ax.plot(t - t[0], y, linewidth=0.9)
        ax.set_title(f"raw {value_col}")
        ax.set_xlabel("seconds from trial start")
        ax.set_ylabel(value_col)
    fig.tight_layout()
    plt.show()

    # 4) intended label과 reported label을 함께 가진 trial-level table을 봅니다.
    answers_eda = load_answers_for_eda(index_eda)
    label_table = answers_eda.select(
        [
            "SubjectID",
            "Trial",
            "NoVideo",
            "IntendedType",
            "IntendedLabel",
            "ReportedType",
            "ReportedLabel",
            "Valence",
            "Arousal",
            "Dominance",
            "Liking",
            "EmotStr",
            "Seen",
        ]
    )
    display(label_table.head(12))

    # 5) intended vs reported count heatmap: 자극 의도와 실제 자기보고가 얼마나 어긋나는지 봅니다.
    counts = answers_eda.group_by(["IntendedType", "ReportedType"]).agg(pl.len().alias("count"))
    matrix = np.zeros((5, 5), dtype=float)
    for intended, reported, count in counts.iter_rows():
        if intended in EDA_EMOTION_NAMES and reported in EDA_EMOTION_NAMES:
            matrix[int(intended) - 1, int(reported) - 1] = count
    labels = [EDA_EMOTION_NAMES[i] for i in range(1, 6)]
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(matrix, cmap="YlGnBu")
    fig.colorbar(im, ax=ax, label="trial count")
    ax.set_xticks(range(5), labels, rotation=45, ha="right")
    ax.set_yticks(range(5), labels)
    ax.set_xlabel("reported label")
    ax.set_ylabel("intended label")
    ax.set_title("Intended label × Reported label")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, int(matrix[i, j]), ha="center", va="center", fontsize=8)
    fig.tight_layout()
    plt.show()

    # 6) rating/label 상관관계 heatmap: Valence, Arousal 등이 label과 어떻게 같이 움직이는지 간단히 확인합니다.
    corr_cols = ["IntendedType", "ReportedType", "Valence", "Arousal", "Dominance", "Liking", "EmotStr", "Seen"]
    corr = answers_eda.select(corr_cols).corr()
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax, label="Pearson correlation")
    ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
    ax.set_yticks(range(len(corr_cols)), corr_cols)
    ax.set_title("Rating / label correlation heatmap")
    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            ax.text(j, i, f"{corr.to_numpy()[i, j]:.2f}", ha="center", va="center", fontsize=7)
    fig.tight_layout()
    plt.show()


## 3. 짧은 PPG feature extractor 정의


In [ ]:
# 짧은 버전용 feature extractor입니다.
# 핵심 아이디어: trial별 CSV를 읽어서 평균/표준편차/분위수/기울기 같은 간단한 통계량으로 바꿉니다.
EMOTION_LABELS = ["neutral", "fun", "sadness", "anger", "fear"]
EMOTION_IDS = [1, 2, 3, 4, 5]
ANSWER_COLS = ["SubjectID", "Trial", "NoVideo", "Valence", "Arousal", "Dominance", "Liking", "EmotType", "EmotStr", "Seen"]
LEAKAGE_COLS = set(ANSWER_COLS + ["IntendedType", "ReportedType"])


def sanitize_name(name: str) -> str:
    name = re.sub(r"[^\w]+", "_", name.strip())
    return re.sub(r"_+", "_", name).strip("_")


def read_csv_clean(path: Path) -> pl.DataFrame:
    # 일부 파일은 인코딩이 섞일 수 있어 utf8-lossy fallback을 둡니다.
    try:
        df = pl.read_csv(path, infer_schema_length=256, ignore_errors=True)
    except Exception:
        df = pl.read_csv(path, encoding="utf8-lossy", infer_schema_length=256, ignore_errors=True)
    return df.rename({c: sanitize_name(c) for c in df.columns})


def as_float_array(values: Any) -> np.ndarray:
    if isinstance(values, pl.Series):
        return values.cast(pl.Float64, strict=False).to_numpy().astype(float, copy=False)
    return np.asarray(values, dtype=float)


def robust_stats(values: Any, prefix: str) -> dict[str, float]:
    x = as_float_array(values)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {f"{prefix}_{k}": math.nan for k in ["mean", "std", "min", "max", "median", "q10", "q90", "rms"]}
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_q10": float(np.quantile(x, 0.10)),
        f"{prefix}_q90": float(np.quantile(x, 0.90)),
        f"{prefix}_rms": float(np.sqrt(np.mean(x**2))),
    }


def time_seconds(values: Any) -> np.ndarray:
    x = as_float_array(values)
    return x if np.isfinite(x).sum() >= 2 else np.arange(len(x), dtype=float)


def slope(time_values: Any, signal_values: Any) -> float:
    t = time_seconds(time_values)
    y = as_float_array(signal_values)
    mask = np.isfinite(t) & np.isfinite(y)
    if mask.sum() < 2:
        return math.nan
    t = t[mask] - np.mean(t[mask])
    y = y[mask] - np.mean(y[mask])
    denom = float(np.dot(t, t))
    return float(np.dot(t, y) / denom) if denom else math.nan


def duration_fs(time_values: Any, n: int, prefix: str) -> dict[str, float]:
    t = time_seconds(time_values)
    t = t[np.isfinite(t)]
    if t.size < 2:
        return {f"{prefix}_duration_s": math.nan, f"{prefix}_fs_hz": math.nan}
    dur = float(np.max(t) - np.min(t))
    fs = float((n - 1) / dur) if dur > 0 and n > 1 else math.nan
    return {f"{prefix}_duration_s": dur, f"{prefix}_fs_hz": fs}


def simple_peak_count(values: Any) -> int:
    # 아주 단순한 peak 수입니다. 정밀 HRV 분석이 아니라 빠른 PPG 형태 확인용입니다.
    y = as_float_array(values)
    y = y[np.isfinite(y)]
    if y.size < 3:
        return 0
    y = y - np.median(y)
    return int(np.sum((y[1:-1] > y[:-2]) & (y[1:-1] >= y[2:]) & (y[1:-1] > np.std(y) * 0.1)))


def add_targets(row: dict[str, Any]) -> dict[str, Any]:
    no_video = int(row.get("NoVideo", 0))
    row["IntendedType"] = 1 if 1 <= no_video <= 10 else 2 if no_video <= 20 else 3 if no_video <= 30 else 4 if no_video <= 40 else 5 if no_video <= 50 else None
    row["ReportedType"] = int(row["EmotType"]) if row.get("EmotType") is not None else None
    return row


def summarize_band(path: Path, mode: str) -> dict[str, Any]:
    df = read_csv_clean(path)
    info: dict[str, Any] = {}
    # PPG는 all/ppg/ppg_rich 모두에서 사용합니다.
    if "PPG" in df.columns:
        info.update(robust_stats(df["PPG"], "Band_PPG"))
        if "PPG_Time" in df.columns:
            info.update(duration_fs(df["PPG_Time"], df.height, "Band_PPG"))
            info["Band_PPG_slope"] = slope(df["PPG_Time"], df["PPG"])
        info["Band_PPG_peak_count"] = simple_peak_count(df["PPG"])
        if mode == "ppg_rich":
            y = as_float_array(df["PPG"])
            info.update(robust_stats(np.diff(y[np.isfinite(y)]), "Band_PPG_diff"))
            for part, segment in enumerate(np.array_split(y, 4), start=1):
                info.update(robust_stats(segment, f"Band_PPG_seg{part}"))
    if mode == "all":
        for col in ["GSR", "SKT"]:
            if col in df.columns:
                info.update(robust_stats(df[col], f"Band_{col}"))
                tcol = f"{col}_Time"
                if tcol in df.columns:
                    info.update(duration_fs(df[tcol], df.height, f"Band_{col}"))
                    info[f"Band_{col}_slope"] = slope(df[tcol], df[col])
    return info


def summarize_resp(path: Path) -> dict[str, Any]:
    df = read_csv_clean(path)
    force = next((c for c in df.columns if c.lower().startswith("force")), None)
    if force is None:
        return {}
    info = robust_stats(df[force], "Resp_Force")
    if "Time" in df.columns:
        info.update(duration_fs(df["Time"], df.height, "Resp"))
        info["Resp_Force_slope"] = slope(df["Time"], df[force])
    return info


def summarize_eeg(path: Path) -> dict[str, Any]:
    df = read_csv_clean(path)
    info: dict[str, Any] = {}
    for col in df.columns:
        if col.lower() in {"timestamp", "time", "ts"}:
            continue
        if df[col].dtype.is_numeric():
            info.update(robust_stats(df[col], f"EEG_{sanitize_name(col)}"))
    return info


def subject_dirs(limit_subjects: int | None) -> list[Path]:
    dirs = sorted([p.parent for p in EXTRACTED_DIR.glob("*/answer.csv")], key=natural_key)
    return dirs[:limit_subjects] if limit_subjects is not None else dirs


def build_feature_table(mode: Literal["all", "ppg", "ppg_rich"], output_path: Path) -> pl.DataFrame:
    rows: list[dict[str, Any]] = []
    subjects = subject_dirs(LIMIT_SUBJECTS)
    if not subjects:
        raise FileNotFoundError("압축해제된 participant 폴더가 없습니다. 다운로드 셀을 먼저 실행하세요.")
    for si, sdir in enumerate(subjects, start=1):
        print(f"[{si}/{len(subjects)}] subject={sdir.name}, mode={mode}")
        answers = read_csv_clean(sdir / "answer.csv")
        for answer in answers.iter_rows(named=True):
            trial = int(answer["Trial"])
            row = {"SubjectID": int(sdir.name), **answer}
            row = add_targets(row)
            band_path = sdir / f"band_{trial}.csv"
            if band_path.exists():
                row.update(summarize_band(band_path, mode))
            if mode == "all":
                resp_path = sdir / f"respiration_{trial}.csv"
                eeg_path = sdir / f"eeg_{trial}.csv"
                if resp_path.exists():
                    row.update(summarize_resp(resp_path))
                if eeg_path.exists():
                    row.update(summarize_eeg(eeg_path))
            rows.append(row)
    frame = pl.DataFrame(rows, infer_schema_length=None)
    first = [c for c in [*ANSWER_COLS, "IntendedType", "ReportedType"] if c in frame.columns]
    frame = frame.select(first + sorted([c for c in frame.columns if c not in first]))
    frame.write_parquet(output_path)
    print(f"saved {output_path}: rows={frame.height}, cols={frame.width}")
    return frame


def ensure_features(mode: str, path: Path) -> pl.DataFrame:
    if SKIP_HEAVY:
        print(f"검증 모드라 feature 생성을 건너뜁니다: {path}")
        return pl.DataFrame()
    if path.exists() and not REBUILD_FEATURES:
        print(f"재사용: {path}")
        return pl.read_parquet(path)
    return build_feature_table(mode, path)


## 4. 학습/평가 함수 정의


In [ ]:
# 짧은 버전용 공통 학습/평가 함수입니다.
# top-1/top-2/top-3, macro F1, balanced accuracy, kappa, confusion matrix를 계산합니다.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold


@dataclass(frozen=True)
class BackendSelection:
    requested: str
    selected: str
    backend: str
    gpu_name: str | None = None
    fallback_reason: str | None = None

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def probe_lightgbm(device_type: str) -> tuple[bool, str | None]:
    try:
        import lightgbm as lgb
        x = np.array([[0, 0], [0, 1], [1, 0], [1, 1], [0.2, 0.1], [0.8, 0.9]], dtype=float)
        y = np.array([0, 0, 1, 1, 0, 1], dtype=int)
        params = {"objective": "multiclass", "num_class": 2, "metric": "multi_logloss", "verbosity": -1, "device_type": device_type, "num_threads": 1}
        lgb.train(params, lgb.Dataset(x, label=y), num_boost_round=1)
        return True, None
    except Exception as exc:
        return False, str(exc).splitlines()[0]


def select_lightgbm_device(requested: str = "auto") -> BackendSelection:
    if requested == "cpu":
        return BackendSelection(requested, "cpu", "lightgbm")
    failures = []
    for candidate in ["cuda", "gpu"]:
        ok, reason = probe_lightgbm(candidate)
        if ok:
            return BackendSelection(requested, candidate, "lightgbm")
        failures.append(f"{candidate}: {reason}")
    return BackendSelection(requested, "cpu", "lightgbm", fallback_reason="; ".join(failures))


def select_feature_columns(frame: pl.DataFrame, feature_set: str) -> list[str]:
    if feature_set == "ppg":
        return [c for c in frame.columns if c.startswith("Band_PPG_") and frame[c].dtype.is_numeric()]
    return [c for c in frame.columns if c not in LEAKAGE_COLS and frame[c].dtype.is_numeric()]


def prepare_matrix(feature_path: Path, feature_set: str, target: str, max_rows: int | None = None):
    frame = pl.read_parquet(feature_path)
    target_col = "ReportedType" if target == "reported" else "IntendedType"
    frame = frame.filter(pl.col(target_col).is_in(EMOTION_IDS))
    if max_rows:
        frame = frame.head(max_rows)
    features = select_feature_columns(frame, feature_set)
    x = frame.select([pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in features]).to_numpy().astype(float)
    y = frame[target_col].to_numpy().astype(int)
    groups = frame["SubjectID"].cast(pl.Utf8).to_numpy()
    return x, y, groups, features


def fold_preprocess(x_train: np.ndarray, x_test: np.ndarray, feature_names: Sequence[str]):
    # train fold 정보만 사용해서 column drop + median imputation을 합니다. test leakage 방지용입니다.
    xtr = x_train.copy(); xte = x_test.copy()
    xtr[~np.isfinite(xtr)] = np.nan; xte[~np.isfinite(xte)] = np.nan
    keep = np.mean(np.isnan(xtr), axis=0) < 0.5
    for i in range(xtr.shape[1]):
        finite = xtr[:, i][np.isfinite(xtr[:, i])]
        if finite.size == 0 or np.unique(finite).size <= 1:
            keep[i] = False
    xtr = xtr[:, keep]; xte = xte[:, keep]
    names = [n for n, k in zip(feature_names, keep, strict=True) if k]
    med = np.nanmedian(xtr, axis=0); med[~np.isfinite(med)] = 0.0
    train_nan = np.isnan(xtr); test_nan = np.isnan(xte)
    xtr[train_nan] = np.take(med, np.where(train_nan)[1])
    xte[test_nan] = np.take(med, np.where(test_nan)[1])
    return xtr, xte, names


def make_model(model_name: str, backend: BackendSelection, seed: int, params: dict[str, Any] | None = None):
    params = dict(params or {})
    if model_name == "lightgbm":
        from lightgbm import LGBMClassifier
        base = dict(n_estimators=350, learning_rate=0.05, num_leaves=63, subsample=0.8, colsample_bytree=0.8, class_weight="balanced", random_state=seed, n_jobs=1, verbosity=-1, device_type=backend.selected)
        base.update(params)
        return LGBMClassifier(**base)
    if model_name == "logistic_regression":
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        base = dict(max_iter=2000, class_weight="balanced", random_state=seed)
        base.update(params)
        return make_pipeline(StandardScaler(), LogisticRegression(**base))
    if model_name == "extra_trees":
        from sklearn.ensemble import ExtraTreesClassifier
        base = dict(n_estimators=500, max_features="sqrt", class_weight="balanced", random_state=seed, n_jobs=-1)
        base.update(params)
        return ExtraTreesClassifier(**base)
    if model_name == "hist_gradient_boosting":
        from sklearn.ensemble import HistGradientBoostingClassifier
        base = dict(learning_rate=0.05, max_iter=250, l2_regularization=0.01, random_state=seed)
        base.update(params)
        return HistGradientBoostingClassifier(**base)
    raise ValueError(model_name)


def aligned_scores(model: Any, x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    pred = np.asarray(model.predict(x), dtype=int)
    proba = np.asarray(model.predict_proba(x), dtype=float)
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "steps"):
        classes = getattr(model.steps[-1][1], "classes_", EMOTION_IDS)
    out = np.zeros((x.shape[0], len(EMOTION_IDS)), dtype=float)
    for j, cls in enumerate(classes):
        if int(cls) in EMOTION_IDS:
            out[:, EMOTION_IDS.index(int(cls))] = proba[:, j]
    return pred, out


def metric_bundle(y_true: Sequence[int], y_pred: Sequence[int], scores: np.ndarray) -> dict[str, Any]:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    order = np.argsort(-scores, axis=1)
    topk = np.asarray(EMOTION_IDS)[order]
    return {
        "top1_accuracy": float(accuracy_score(y_true, y_pred)),
        "top2_accuracy": float(np.mean([yt in row[:2] for yt, row in zip(y_true, topk, strict=True)])),
        "top3_accuracy": float(np.mean([yt in row[:3] for yt, row in zip(y_true, topk, strict=True)])),
        "macro_f1": float(f1_score(y_true, y_pred, labels=EMOTION_IDS, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=EMOTION_IDS, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred, labels=EMOTION_IDS)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=EMOTION_IDS).tolist(),
    }


def run_cv(feature_path: Path, *, model_name: str, feature_set: str, target: str = "reported", split: str = "grouped", params: dict[str, Any] | None = None) -> dict[str, Any]:
    x, y, groups, features = prepare_matrix(feature_path, feature_set, target)
    n_splits = min(FOLDS, len(np.unique(groups))) if split == "grouped" else FOLDS
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED) if split == "grouped" else StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    backend = select_lightgbm_device(DEVICE) if model_name == "lightgbm" else BackendSelection(DEVICE, "cpu", model_name)
    ys, ps, ss = [], [], []
    for fold, (tr, te) in enumerate(splitter.split(x, y, groups if split == "grouped" else None), start=1):
        xtr, xte, kept = fold_preprocess(x[tr], x[te], features)
        model = make_model(model_name, backend, SEED + fold, params)
        model.fit(xtr, y[tr])
        pred, score = aligned_scores(model, xte)
        ys.extend(y[te].tolist()); ps.extend(pred.tolist()); ss.append(score)
        print(f"fold {fold}: test_rows={len(te)}, kept_features={len(kept)}")
    metrics = metric_bundle(ys, ps, np.vstack(ss))
    return {"model_name": model_name, "feature_path": str(feature_path), "feature_set": feature_set, "target": target, "split": split, "backend": backend.to_dict(), "rows": int(len(y)), "features": int(len(features)), "metrics": metrics}


def save_json(obj: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))


## 5. GPU/CPU backend 확인


In [ ]:
backend = select_lightgbm_device(DEVICE)
print(f"요청 DEVICE: {backend.requested}")
print(f"선택된 LightGBM device_type: {backend.selected}")
if backend.fallback_reason:
    print("fallback 이유:", backend.fallback_reason)


## 6. PPG / rich-PPG feature 생성


In [ ]:
ppg = ensure_features("ppg", FEATURE_PPG)
ppg_rich = ensure_features("ppg_rich", FEATURE_PPG_RICH)
print("ppg:", ppg.shape)
print("ppg_rich:", ppg_rich.shape)


## 7. 대안모델 baseline 비교


In [ ]:
if SKIP_HEAVY:
    baseline_results = {}
else:
    baseline_results = {}
    for model_name in ["logistic_regression", "lightgbm", "extra_trees", "hist_gradient_boosting"]:
        out = RESULT_DIR / f"{model_name}_ppg.json"
        if out.exists() and not RERUN_BASELINES:
            result = json.loads(out.read_text())
            print(f"재사용: {out}")
        else:
            print(f"실행: {model_name}/ppg")
            result = run_cv(FEATURE_PPG, model_name=model_name, feature_set="ppg", target="reported", split="grouped")
            save_json(result, out)
        baseline_results[f"{model_name}_ppg"] = result

    for model_name in ["logistic_regression", "lightgbm"]:
        out = RESULT_DIR / f"{model_name}_ppg_rich.json"
        if out.exists() and not RERUN_BASELINES:
            result = json.loads(out.read_text())
        else:
            print(f"실행: {model_name}/ppg_rich")
            result = run_cv(FEATURE_PPG_RICH, model_name=model_name, feature_set="ppg", target="reported", split="grouped")
            save_json(result, out)
        baseline_results[f"{model_name}_ppg_rich"] = result


## 8. Optuna 짧은 탐색


In [ ]:
# Optuna 탐색 공간도 노트북 안에 짧게 둡니다.
# trial 수가 작으면 빠르지만 불안정합니다. 논문용 탐색에서는 OPTUNA_TRIALS를 크게 늘리세요.
import optuna


def sample_params(trial: optuna.Trial, model_name: str) -> dict[str, Any]:
    if model_name == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 120, 600),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        }
    if model_name == "logistic_regression":
        return {"C": trial.suggest_float("C", 1e-3, 100.0, log=True), "solver": "lbfgs"}
    raise ValueError(model_name)


def tune_model(model_name: str, feature_path: Path) -> dict[str, Any]:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))

    def objective(trial: optuna.Trial) -> float:
        result = run_cv(feature_path, model_name=model_name, feature_set="ppg", target="reported", split="grouped", params=sample_params(trial, model_name))
        trial.set_user_attr("metrics", result["metrics"])
        return float(result["metrics"]["macro_f1"])

    study.optimize(objective, n_trials=OPTUNA_TRIALS, gc_after_trial=True)
    best = run_cv(feature_path, model_name=model_name, feature_set="ppg", target="reported", split="grouped", params=dict(study.best_trial.params))
    out = OPTUNA_DIR / f"{model_name}_ppg_best.json"
    save_json({"best_params": dict(study.best_trial.params), "best_result": best}, out)
    print(f"{model_name} best macro_f1={best['metrics']['macro_f1']:.4f}, saved={out}")
    return {"model_name": model_name, "best_params": dict(study.best_trial.params), "best_result": best}


In [ ]:
if SKIP_HEAVY:
    optuna_results = []
elif RERUN_OPTUNA or not list(OPTUNA_DIR.glob("*_ppg_best.json")):
    optuna_results = [tune_model("logistic_regression", FEATURE_PPG), tune_model("lightgbm", FEATURE_PPG)]
else:
    optuna_results = []
    for path in sorted(OPTUNA_DIR.glob("*_ppg_best.json")):
        payload = json.loads(path.read_text())
        optuna_results.append({"model_name": path.name.replace("_ppg_best.json", ""), **payload})
        print(f"Optuna 결과 재사용: {path}")


## 9. 결과 요약 그래프


In [ ]:
rows = []
for name, result in baseline_results.items():
    m = result["metrics"]
    rows.append({"run": name, "kind": "fixed", "backend": result["backend"]["selected"], "top1": round(m["top1_accuracy"], 4), "macro_f1": round(m["macro_f1"], 4), "balanced_acc": round(m["balanced_accuracy"], 4)})
for item in optuna_results:
    result = item["best_result"]
    m = result["metrics"]
    rows.append({"run": f"optuna_{item['model_name']}", "kind": "optuna", "backend": result["backend"]["selected"], "top1": round(m["top1_accuracy"], 4), "macro_f1": round(m["macro_f1"], 4), "balanced_acc": round(m["balanced_accuracy"], 4)})
comparison = pl.DataFrame(rows).sort("macro_f1", descending=True) if rows else pl.DataFrame()
display(comparison)

if comparison.height:
    plot_df = comparison.sort("macro_f1")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(plot_df["run"].to_list(), plot_df["macro_f1"].to_numpy())
    ax.set_xlim(0, 1); ax.set_xlabel("macro F1"); ax.set_title("PPG 대안모델 비교")
    fig.tight_layout(); plt.show()
